<div style="display:flex; align-items:center; justify-content:space-between; background:linear-gradient(135deg,#0F0F1A 0%,#1A1A2E 100%); padding:28px 36px; border-radius:14px; margin-bottom:16px;"><div><h1 style="color:white; margin:0; font-size:2em; font-weight:900;">Fundamentos de Data Science</h1><h2 style="color:#6C3CE1; margin:6px 0 0; font-size:1.3em; font-weight:700;">Practica para el Examen -- Limpieza y EDA</h2><p style="color:#888; margin:8px 0 0;">Empresa de tecnologia movil -- Analisis de retencion de usuarios</p></div><img src="images/LOGO-CORREO_x.png" width="130"/></div>

## Contexto del Problema

Una compania de tecnologia emergente que desarrolla aplicaciones moviles quiere **mejorar la experiencia del usuario y aumentar la retencion**. Han recolectado datos sobre el uso de sus aplicaciones y te piden que limpies y explores los datos.

## Diccionario de Datos

| Columna | Tipo | Descripcion |
|---------|------|-------------|
| `user_id` | int | Identificacion unica del usuario |
| `app_name` | str | Nombre de la aplicacion |
| `app_version` | str | Version de la aplicacion usada |
| `platform` | str | Plataforma del dispositivo (Android, iOS) |
| `device_type` | str | Tipo de dispositivo (Phone, Tablet) |
| `session_duration` | float | Duracion de la sesion en minutos |
| `number_of_sessions` | float | Numero de sesiones en un dia |
| `country` | str | Pais del usuario |
| `user_feedback` | float | Puntuacion de experiencia (1-5) |
| `age` | float | Edad del usuario |
| `subscription_type` | str | Tipo de suscripcion (Free, Premium) |

---

<div style="background:linear-gradient(135deg,#6C3CE1 0%,#1A1A2E 100%); padding:22px 32px; border-radius:12px; margin:28px 0 16px;"><h2 style="color:white; margin:0; font-size:1.5em; font-weight:900;">Generacion del Dataset Simulamos datos reales con errores tipicos de produccion</h2><p style="color:rgba(255,255,255,0.82); margin:7px 0 0;">#3B1F8C</p></div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0F0F1A',
    'axes.facecolor':   '#1A1A2E',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#CCC',
    'xtick.color':      '#AAA',
    'ytick.color':      '#AAA',
    'text.color':       'white',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'legend.facecolor': '#1A1A2E',
    'legend.edgecolor': '#444',
})
PURPLE = '#6C3CE1'
TEAL   = '#00C9A7'
AMBER  = '#F59E0B'
RED    = '#E44D26'


In [ ]:
import random
np.random.seed(42)
random.seed(42)

n = 500
apps = ['AppVibe', 'TaskFlow', 'QuickNotes', 'FitTrack', 'ShopEasy',
        'MoodBoard', 'LearnHub', 'SocialSnap', 'FinTrack', 'TravelBuddy']

# Patrones realistas: Premium -> sesiones mas largas; iOS -> mejor feedback
is_premium = np.random.choice([False, True], n, p=[0.65, 0.35])
is_ios     = np.random.choice([False, True], n, p=[0.58, 0.42])

df = pd.DataFrame({
    'user_id':            range(1, n + 1),
    'app_name':           np.random.choice(apps, n),
    'app_version':        np.random.choice(['1.0', '1.1', '1.2', '2.0', '2.1'], n),
    'platform':           np.where(is_ios, 'iOS', 'Android'),
    'device_type':        np.random.choice(['Phone', 'Tablet'], n, p=[0.78, 0.22]),
    'session_duration':   np.where(is_premium,
                              np.random.randint(30, 120, n),
                              np.random.randint(5,  60,  n)).astype(float),
    'number_of_sessions': np.random.randint(1, 15, n).astype(float),
    'country':            np.random.choice(['USA','Canada','Mexico','UK',
                                            'Germany','France','Spain',
                                            'Chile','Brazil','Colombia'], n),
    'user_feedback':      np.where(is_ios,
                              np.random.randint(3, 6, n),
                              np.random.randint(1, 6, n)).astype(float),
    'age':                np.random.randint(18, 65, n).astype(float),
    'subscription_type':  np.where(is_premium, 'Premium', 'Free'),
})

# --- Errores tipicos de produccion ---

# 1. Inconsistencias categoricas
plat_v = {'Android': ['android','ANDROID','Android'],
          'iOS':     ['ios',    'IOS',    'iOS']}
subs_v = {'Free':    ['free',   'FREE',   'Free'],
          'Premium': ['premium','PREMIUM','Premium']}
for i in random.sample(range(n), 70):
    df.loc[i, 'platform'] = random.choice(plat_v[df.loc[i, 'platform']])
for i in random.sample(range(n), 50):
    df.loc[i, 'subscription_type'] = random.choice(subs_v[df.loc[i, 'subscription_type']])

# 2. Outliers fisicamente imposibles
for i in random.sample(range(n), 18):
    df.loc[i, 'session_duration'] = random.choice([480, 600, 720, 999, 1440, 2880])
for i in random.sample(range(n), 12):
    df.loc[i, 'number_of_sessions'] = random.choice([40, 60, 80, 100, 150])

# 3. Nulos en columnas clave
df.loc[random.sample(range(n), 45), 'user_feedback']    = np.nan   # ~9%
df.loc[random.sample(range(n), 38), 'age']              = np.nan   # ~7.6%
df.loc[random.sample(range(n), 22), 'session_duration'] = np.nan   # ~4.4%
df.loc[random.sample(range(n), 12), 'country']          = np.nan   # ~2.4%

# 4. Duplicados (25 filas repetidas, mezcladas)
dupes = df.sample(25, random_state=7)
df = pd.concat([df, dupes], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv('user_app_data.csv', index=False)
print(f'Dataset guardado: {df.shape[0]} filas x {df.shape[1]} columnas')
print('(incluye ~25 duplicados, inconsistencias, nulos y outliers)')


<div style="background:linear-gradient(135deg,#6C3CE1 0%,#1A1A2E 100%); padding:22px 32px; border-radius:12px; margin:28px 0 16px;"><h2 style="color:white; margin:0; font-size:1.5em; font-weight:900;">Seccion 1 -- Carga y Primera Exploracion head | tail | info | describe | shape</h2><p style="color:rgba(255,255,255,0.82); margin:7px 0 0;">#00503E</p></div>

<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 1.1</span><span style="color:white; font-weight:600; margin-left:10px;">Cargar el dataset</span></div>

In [ ]:
df = pd.read_csv('user_app_data.csv')
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Memoria aproximada: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 1.2</span><span style="color:white; font-weight:600; margin-left:10px;">Primeras y ultimas filas</span></div>

In [ ]:
print('=== Primeras 5 filas (head) ===')
display(df.head())
print('\n=== Ultimas 5 filas (tail) ===')
display(df.tail())


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 1.3</span><span style="color:white; font-weight:600; margin-left:10px;">Estructura y tipos de datos</span></div>

In [ ]:
print('=== Informacion del DataFrame (info) ==='); df.info()


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 1.4</span><span style="color:white; font-weight:600; margin-left:10px;">Estadisticas descriptivas</span></div>

In [ ]:
print('=== Estadisticas columnas numericas ===')
display(df.describe().round(2))
print('\n=== Estadisticas columnas categoricas ===')
display(df.describe(include='object'))


<div style="background:linear-gradient(135deg,#6C3CE1 0%,#1A1A2E 100%); padding:22px 32px; border-radius:12px; margin:28px 0 16px;"><h2 style="color:white; margin:0; font-size:1.5em; font-weight:900;">Seccion 2 -- Limpieza de Datos tipos | duplicados | inconsistencias | nulos | outliers</h2><p style="color:rgba(255,255,255,0.82); margin:7px 0 0;">#5C1A00</p></div>

<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.1</span><span style="color:white; font-weight:600; margin-left:10px;">Verificar tipos de datos</span></div>

In [ ]:
print('Tipos de datos actuales:')
print(df.dtypes)
print()
tipos_esperados = {
    'user_id': 'int/float', 'app_name': 'object', 'app_version': 'object',
    'platform': 'object', 'device_type': 'object', 'session_duration': 'float',
    'number_of_sessions': 'float', 'country': 'object', 'user_feedback': 'float',
    'age': 'float', 'subscription_type': 'object'
}
print('Verificacion vs diccionario de datos:')
for col, te in tipos_esperados.items():
    tr = str(df[col].dtype)
    ok = '  OK' if te.split('/')[0] in tr else '  REVISAR'
    print(f'{ok}  {col:<22} esperado: {te:<12}  actual: {tr}')


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.2</span><span style="color:white; font-weight:600; margin-left:10px;">Identificar y eliminar duplicados</span></div>

In [ ]:
n_total = len(df)
n_dupes = df.duplicated().sum()
print(f'Total de filas:   {n_total}')
print(f'Filas duplicadas: {n_dupes}  ({n_dupes/n_total*100:.1f}% del dataset)')
print()
print('Ejemplo de fila duplicada:')
display(df[df.duplicated(keep=False)].sort_values('user_id')
          [['user_id','app_name','platform','session_duration','user_feedback']].head(4))
df = df.drop_duplicates().reset_index(drop=True)
print(f'Duplicados eliminados. Filas restantes: {len(df)}')


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.3</span><span style="color:white; font-weight:600; margin-left:10px;">Corregir inconsistencias en valores categoricos</span></div>

In [ ]:
print('Valores unicos ANTES de limpiar:')
print(f'  platform:          {sorted(df["platform"].dropna().unique())}')
print(f'  subscription_type: {sorted(df["subscription_type"].dropna().unique())}')
print()
df['platform']          = df['platform'].str.strip().str.lower()
df['platform']          = df['platform'].map({'android': 'Android', 'ios': 'iOS'})
df['subscription_type'] = df['subscription_type'].str.strip().str.capitalize()
df['device_type']       = df['device_type'].str.strip().str.capitalize()
print('Valores unicos DESPUES de limpiar:')
print(f'  platform:          {sorted(df["platform"].dropna().unique())}')
print(f'  subscription_type: {sorted(df["subscription_type"].dropna().unique())}')
print(f'  device_type:       {sorted(df["device_type"].dropna().unique())}')
print('Inconsistencias corregidas')


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.4</span><span style="color:white; font-weight:600; margin-left:10px;">Analisis de valores nulos</span></div>

In [ ]:
null_report = pd.DataFrame({
    'Nulos':      df.isnull().sum(),
    'Porcentaje': (df.isnull().sum() / len(df) * 100).round(2)
}).query('Nulos > 0').sort_values('Porcentaje', ascending=False)
null_report['Umbral'] = null_report['Porcentaje'].apply(
    lambda x: 'Imputar (< 30%)' if x < 30 else
              ('Evaluar (30-50%)' if x < 50 else 'Eliminar columna (> 50%)')
)
print('=== Columnas con valores nulos ===')
display(null_report)


### Estrategia de Imputacion por Columna

| Columna | Nulos% | Estrategia | Justificacion |
|---------|--------|------------|---------------|
| `user_feedback` | ~9% | **Mediana por `platform`** | iOS y Android tienen tendencias de rating distintas; usar la mediana del grupo preserva esa diferencia sin introducir sesgos extremos |
| `age` | ~7.6% | **Mediana por `country`** | La demografia varia por region; imputar con la edad mediana del mismo pais es mas preciso que la mediana global |
| `session_duration` | ~4.4% | **Mediana por `subscription_type`** | Los usuarios Premium tienen sesiones mas largas; usar el grupo correcto evita subestimar/sobreestimar |
| `country` | ~2.4% | **Categoria 'Unknown'** | Variable categorica sin relacion inferible con otras columnas; inventar un pais seria incorrecto |

> **Regla de umbral usada:**
> - < 30% nulos: imputar las filas
> - 30-50%: evaluar si la columna es critica antes de imputar o eliminar
> - > 50%: eliminar la columna

<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.5</span><span style="color:white; font-weight:600; margin-left:10px;">Imputar user_feedback -- mediana por plataforma</span></div>

In [ ]:
mediana_fb = df.groupby('platform')['user_feedback'].median()
print('Mediana de user_feedback por plataforma:')
print(mediana_fb.to_string())
df['user_feedback'] = df.groupby('platform')['user_feedback'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Nulos restantes en user_feedback: {df["user_feedback"].isnull().sum()}')


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.6</span><span style="color:white; font-weight:600; margin-left:10px;">Imputar age -- mediana por pais</span></div>

In [ ]:
mediana_age = df.groupby('country')['age'].median().round(1)
print('Mediana de age por pais:')
print(mediana_age.to_string())
df['age'] = df.groupby('country')['age'].transform(lambda x: x.fillna(x.median()))
if df['age'].isnull().sum() > 0:
    df['age'] = df['age'].fillna(df['age'].median())
df['age'] = df['age'].round(0).astype(int)
print(f'Nulos restantes en age: {df["age"].isnull().sum()}')


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.7</span><span style="color:white; font-weight:600; margin-left:10px;">Imputar session_duration -- mediana por tipo de suscripcion</span></div>

In [ ]:
mediana_dur = df.groupby('subscription_type')['session_duration'].median()
print('Mediana de session_duration por tipo de suscripcion:')
print(mediana_dur.to_string())
df['session_duration'] = df.groupby('subscription_type')['session_duration'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Nulos restantes en session_duration: {df["session_duration"].isnull().sum()}')


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.8</span><span style="color:white; font-weight:600; margin-left:10px;">Tratar nulos en country -- categoria Unknown</span></div>

In [ ]:
nc = df['country'].isnull().sum()
print(f'Nulos en country: {nc} ({nc/len(df)*100:.1f}%)')
print('Estrategia: variable categorica sin relacion inferible -> asignamos Unknown')
df['country'] = df['country'].fillna('Unknown')
print(f'Nulos restantes: {df["country"].isnull().sum()}')


<div style="background:#1A1A2E; border-left:5px solid #6C3CE1; padding:12px 18px; border-radius:6px; margin:18px 0 8px;"><span style="color:#6C3CE1; font-weight:700;">Paso 2.9</span><span style="color:white; font-weight:600; margin-left:10px;">Identificar y tratar outliers</span></div>

In [ ]:
print('=' * 58)
print('DETECCION DE OUTLIERS -- Metodo IQR + Conocimiento del Dominio')
print('=' * 58)

checks = [
    ('session_duration',   240, 'min -- maximo razonable: 4h de uso movil en un dia'),
    ('number_of_sessions', 30,  'ses -- maximo razonable: 30 sesiones/dia'),
]

for col, lim_dom, nota in checks:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr    = q3 - q1
    lim_iqr = q3 + 1.5 * iqr
    out_iqr = df[df[col] > lim_iqr]
    out_dom = df[df[col] > lim_dom]
    print(f'\nColumna: {col}')
    print(f'  Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}  Limite IQR={lim_iqr:.1f}')
    print(f'  Outliers por IQR:     {len(out_iqr)} filas ({len(out_iqr)/len(df)*100:.1f}%)')
    print(f'  Outliers por dominio: {len(out_dom)} filas ({len(out_dom)/len(df)*100:.1f}%)')
    print(f'  Valores: {sorted(out_dom[col].unique())[:8]}')
    print(f'  Nota: {nota}')
    print('  Decision: eliminar filas (< 5% del total, valores fisicamente imposibles)')


In [ ]:
filas_antes = len(df)
df = df[(df['session_duration']   <= 240) &
        (df['number_of_sessions'] <= 30)].copy().reset_index(drop=True)
eliminadas = filas_antes - len(df)
print(f'Filas eliminadas por outliers: {eliminadas} ({eliminadas/filas_antes*100:.1f}%)')
print(f'Dataset limpio final: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Nulos totales: {df.isnull().sum().sum()}')
print()
print('Estadisticas columnas numericas tras limpieza:')
display(df[['session_duration','number_of_sessions','user_feedback','age']].describe().round(2))


<div style="background:linear-gradient(135deg,#6C3CE1 0%,#1A1A2E 100%); padding:22px 32px; border-radius:12px; margin:28px 0 16px;"><h2 style="color:white; margin:0; font-size:1.5em; font-weight:900;">Seccion 3 -- Exploracion de Datos (EDA) Univariante x2  |  Multivariante x2  |  Interpretacion a profundidad</h2><p style="color:rgba(255,255,255,0.82); margin:7px 0 0;">#004080</p></div>

### Visualizacion Univariante 1 -- Distribucion de Duracion de Sesion

**Objetivo:** Entender como se distribuye el tiempo que los usuarios pasan en cada sesion y detectar si existe asimetria o agrupaciones.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribucion de Duracion de Sesion (minutos)',
             fontsize=14, fontweight='bold', color='white', y=1.01)

media   = df['session_duration'].mean()
mediana = df['session_duration'].median()

# Histograma
axes[0].hist(df['session_duration'], bins=35, color=PURPLE,
             edgecolor='#0F0F1A', alpha=0.9, linewidth=0.5)
axes[0].axvline(media,   color=TEAL,  lw=2.5, ls='--', label=f'Media: {media:.1f} min')
axes[0].axvline(mediana, color=AMBER, lw=2.5, ls=':',  label=f'Mediana: {mediana:.1f} min')
axes[0].set_title('Histograma', color='white')
axes[0].set_xlabel('Duracion (min)')
axes[0].set_ylabel('Cantidad de usuarios')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Densidad KDE
sns.kdeplot(data=df, x='session_duration', ax=axes[1],
            color=PURPLE, fill=True, alpha=0.45, linewidth=2)
axes[1].axvline(media,   color=TEAL,  lw=2.5, ls='--', label=f'Media: {media:.1f}')
axes[1].axvline(mediana, color=AMBER, lw=2.5, ls=':',  label=f'Mediana: {mediana:.1f}')
axes[1].set_title('Densidad (KDE)', color='white')
axes[1].set_xlabel('Duracion (min)')
axes[1].set_ylabel('Densidad')
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Media:       {media:.2f} min')
print(f'Mediana:     {mediana:.2f} min')
print(f'Std:         {df["session_duration"].std():.2f} min')
print(f'Asimetria:   {df["session_duration"].skew():.3f}')
print(f'Curtosis:    {df["session_duration"].kurt():.3f}')


<div style="background:#0D1F0D; border-left:4px solid #00C9A7; padding:14px 18px; border-radius:6px; margin:12px 0;"><p style="color:#00C9A7; font-weight:700; margin:0 0 4px;">Interpretacion</p><p style="color:#CCC; margin:0;">La distribucion de duracion de sesion muestra una <strong>asimetria positiva (cola a la derecha)</strong>: la mayoria de los usuarios tienen sesiones cortas (5-40 min), pero existe un segmento con sesiones muy largas (60-120 min) que eleva la media por encima de la mediana. Esto indica la coexistencia de dos perfiles de usuario distintos: usuarios casuales (sesiones breves) y usuarios comprometidos (probablemente Premium). La empresa deberia analizar que contenido o funcionalidades generan esas sesiones largas para replicar ese comportamiento en el segmento casual.</p></div>

### Visualizacion Univariante 2 -- Distribucion por Plataforma y Suscripcion

**Objetivo:** Conocer el mix de usuarios por plataforma y tipo de suscripcion para entender la composicion de la base.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribucion de Variables Categoricas Clave',
             fontsize=14, fontweight='bold', color='white')

# Barras: plataforma
plat = df['platform'].value_counts()
bars = axes[0].bar(plat.index, plat.values,
                   color=[PURPLE, TEAL], edgecolor='#0F0F1A', width=0.55, linewidth=1.2)
for bar, val in zip(bars, plat.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
                 f'{val}\n({val/len(df)*100:.1f}%)',
                 ha='center', fontsize=11, color='white', fontweight='bold')
axes[0].set_title('Usuarios por Plataforma', color='white')
axes[0].set_ylabel('Cantidad de Usuarios')
axes[0].set_ylim(0, plat.max() * 1.25)
axes[0].grid(axis='y', alpha=0.3)

# Pie: suscripcion
subs = df['subscription_type'].value_counts()
wedges, texts, autotexts = axes[1].pie(
    subs.values, labels=subs.index, autopct='%1.1f%%',
    colors=[TEAL, PURPLE], startangle=90,
    wedgeprops={'edgecolor': '#0F0F1A', 'linewidth': 2.5},
    textprops={'color': 'white', 'fontsize': 12}
)
for at in autotexts:
    at.set_color('white'); at.set_fontweight('bold')
axes[1].set_title('Distribucion por Tipo de Suscripcion', color='white')

plt.tight_layout()
plt.show()

print('Por plataforma:')
print((df['platform'].value_counts(normalize=True)*100).round(1).to_string())
print('\nPor suscripcion:')
print((df['subscription_type'].value_counts(normalize=True)*100).round(1).to_string())


<div style="background:#0D1F0D; border-left:4px solid #00C9A7; padding:14px 18px; border-radius:6px; margin:12px 0;"><p style="color:#00C9A7; font-weight:700; margin:0 0 4px;">Interpretacion</p><p style="color:#CCC; margin:0;">Android representa la mayoria de la base de usuarios (~58%), coherente con la cuota global de mercado. Aproximadamente un <strong>35% de los usuarios es Premium</strong>, una tasa de conversion solida para una app emergente. Este segmento es estrategico: si los usuarios Premium tienen mayor retencion y feedback positivo, la empresa deberia enfocar campanas de conversion hacia los usuarios Free de mayor engagement. La mayoria de usuarios Android son Free, lo que representa la mayor oportunidad de monetizacion.</p></div>

### Visualizacion Multivariante 1 -- Duracion vs Frecuencia de Sesiones por Suscripcion

**Objetivo:** Explorar si existe relacion entre el tiempo de sesion y la frecuencia diaria de uso, segmentando por tipo de suscripcion para detectar patrones de comportamiento diferenciados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Duracion de Sesion vs Numero de Sesiones Diarias',
             fontsize=14, fontweight='bold', color='white')

colores = {'Free': PURPLE, 'Premium': TEAL}

# Scatter con tendencia
for subs_label, group in df.groupby('subscription_type'):
    axes[0].scatter(group['session_duration'], group['number_of_sessions'],
                    alpha=0.4, s=45, color=colores[subs_label], label=subs_label)
    z = np.polyfit(group['session_duration'], group['number_of_sessions'], 1)
    x_line = np.linspace(group['session_duration'].min(),
                         group['session_duration'].max(), 200)
    axes[0].plot(x_line, np.poly1d(z)(x_line),
                 color=colores[subs_label], lw=2.5, ls='--', alpha=0.9)
axes[0].set_title('Scatter con Linea de Tendencia', color='white')
axes[0].set_xlabel('Duracion de Sesion (min)')
axes[0].set_ylabel('Sesiones por Dia')
axes[0].legend(title='Suscripcion', fontsize=10)
axes[0].grid(alpha=0.2)

# Hexbin: densidad
hb = axes[1].hexbin(df['session_duration'], df['number_of_sessions'],
                    gridsize=22, cmap='RdPu', mincnt=1)
fig.colorbar(hb, ax=axes[1], label='Conteo de usuarios')
axes[1].set_title('Mapa de Densidad (Hexbin)', color='white')
axes[1].set_xlabel('Duracion de Sesion (min)')
axes[1].set_ylabel('Sesiones por Dia')

plt.tight_layout()
plt.show()

print('Correlacion session_duration ~ number_of_sessions:')
for label, grp in df.groupby('subscription_type'):
    r = grp['session_duration'].corr(grp['number_of_sessions'])
    print(f'  {label}: r = {r:.3f}')
print(f'  Global:  r = {df["session_duration"].corr(df["number_of_sessions"]):.3f}')


<div style="background:#0D1F0D; border-left:4px solid #00C9A7; padding:14px 18px; border-radius:6px; margin:12px 0;"><p style="color:#00C9A7; font-weight:700; margin:0 0 4px;">Interpretacion</p><p style="color:#CCC; margin:0;">La correlacion entre duracion y frecuencia de sesiones es <strong>muy debil en ambos grupos</strong>, lo que indica que sesiones largas no implican mas sesiones diarias y viceversa: son comportamientos independientes. Los usuarios <strong>Premium se concentran visiblemente en la zona de sesiones largas</strong> (40-120 min), mientras que los Free se dispersan en sesiones cortas y frecuentes (5-30 min, 1-8 veces/dia). El mapa hexbin confirma que la <strong>mayor densidad global</strong> se encuentra en sesiones de 10-60 min con 1-8 sesiones diarias. Recomendacion: identificar que funcionalidades generan sesiones largas en Premium y exponerlas gradualmente a usuarios Free para incentivar la conversion.</p></div>

### Visualizacion Multivariante 2 -- Feedback por Plataforma y Tipo de Suscripcion

**Objetivo:** Comparar la calidad de la experiencia entre plataformas y tipos de suscripcion para identificar segmentos insatisfechos y priorizar mejoras.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Distribucion del User Feedback por Segmento',
             fontsize=14, fontweight='bold', color='white')

# BoxPlot: feedback por plataforma
sns.boxplot(
    data=df, x='platform', y='user_feedback', ax=axes[0],
    palette={'Android': PURPLE, 'iOS': TEAL},
    width=0.5, linewidth=1.5,
    flierprops=dict(marker='o', markersize=4, alpha=0.4)
)
for i, plat in enumerate(df['platform'].unique()):
    m = df[df['platform'] == plat]['user_feedback'].mean()
    axes[0].plot(i, m, 'D', color='white', markersize=9, zorder=5,
                 label=f'Media {plat}: {m:.2f}')
axes[0].set_title('Feedback por Plataforma', color='white')
axes[0].set_xlabel('Plataforma')
axes[0].set_ylabel('Puntuacion (1-5)')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.25)
axes[0].set_ylim(0.5, 5.5)

# Violin: feedback por suscripcion y plataforma
sns.violinplot(
    data=df, x='subscription_type', y='user_feedback', hue='platform',
    ax=axes[1], palette={'Android': PURPLE, 'iOS': TEAL},
    split=False, inner='quart', linewidth=1.2, alpha=0.8
)
axes[1].set_title('Feedback por Suscripcion y Plataforma (Violin)', color='white')
axes[1].set_xlabel('Tipo de Suscripcion')
axes[1].set_ylabel('Puntuacion (1-5)')
axes[1].legend(title='Plataforma', fontsize=10)
axes[1].grid(axis='y', alpha=0.25)
axes[1].set_ylim(0.5, 5.5)

plt.tight_layout()
plt.show()

print('Estadisticas de feedback por segmento:')
tabla = df.groupby(['platform', 'subscription_type'])['user_feedback'].agg(
    Media='mean', Mediana='median', Std='std'
).round(2)
display(tabla)


<div style="background:#0D1F0D; border-left:4px solid #00C9A7; padding:14px 18px; border-radius:6px; margin:12px 0;"><p style="color:#00C9A7; font-weight:700; margin:0 0 4px;">Interpretacion</p><p style="color:#CCC; margin:0;">Los usuarios <strong>iOS tienen un feedback consistentemente mas alto</strong> que Android en ambos tipos de suscripcion, lo que puede reflejar mejor rendimiento de la app en iOS o un perfil de usuario distinto. Los usuarios <strong>Premium muestran mayor satisfaccion que los Free</strong> en ambas plataformas, validando el valor percibido de la suscripcion de pago. El grafico de violin revela que los usuarios <strong>Android Free tienen la distribucion mas ancha</strong> (mayor variabilidad en el feedback), senalando un grupo heterogeneo con usuarios tanto muy satisfechos como muy insatisfechos. <strong>Prioridad accionable:</strong> investigar y corregir los puntos de dolor especificos en Android antes de las proximas versiones, especialmente en la experiencia Free.</p></div>

<div style="background:linear-gradient(135deg,#6C3CE1 0%,#1A1A2E 100%); padding:22px 32px; border-radius:12px; margin:28px 0 16px;"><h2 style="color:white; margin:0; font-size:1.5em; font-weight:900;">Resumen del Proceso Que se limpio, que se encontro y que se recomienda</h2><p style="color:rgba(255,255,255,0.82); margin:7px 0 0;">#1A2E1A</p></div>

In [ ]:
print('=' * 58)
print('RESUMEN DEL ANALISIS')
print('=' * 58)

resumen = f"""
Dataset original:    525 filas x 11 columnas

Limpieza realizada:
  Duplicados eliminados:   25 filas
  Inconsistencias:         platform y subscription_type normalizados
  Nulos imputados:
      user_feedback    ->  mediana por plataforma
      age              ->  mediana por pais
      session_duration ->  mediana por tipo de suscripcion
      country          ->  categoria 'Unknown'
  Outliers eliminados:     valores fisicamente imposibles
      session_duration > 240 min
      number_of_sessions > 30

Dataset limpio: {len(df)} filas x {df.shape[1]} columnas
Nulos totales: {df.isnull().sum().sum()}

Hallazgos EDA:
  1. Duracion de sesion con asimetria positiva: existe un segmento
     comprometido (sesiones largas) diferente al usuario casual
  2. Android ~58% de usuarios; ~35% son Premium
  3. Correlacion muy debil entre duracion y frecuencia de sesiones
  4. iOS + Premium = mayor satisfaccion en todos los segmentos
  5. Android Free: segmento mas heterogeneo y mayor oportunidad

Recomendaciones:
  -> Priorizar mejoras en la app Android (especialmente Free)
  -> Analizar que genera sesiones largas en Premium para replicarlo
  -> Disenar flujo de conversion Free->Premium basado en el uso
"""
print(resumen)


---
<div style="text-align:center; padding:14px; color:#888; font-size:0.85em; border-top:1px solid #333; margin-top:20px;">SKILLNEST -- Practica para Examen -- Fundamentos de Data Science</div>